In [5]:
import re
# Import the 're' module for regular expression operations.

import os
# Import the 'os' module to interact with the operating system's file system.

import time
# Import the 'time' module to pause execution in the main loop.

from watchdog.observers import Observer
# Import the 'Observer' class from the 'watchdog.observers' module.
# The Observer watches for file system events and dispatches them to event handlers.

from watchdog.events import FileSystemEventHandler
# Import the 'FileSystemEventHandler' class to create custom event handlers.

import tkinter as tk
from tkinter import simpledialog, messagebox
# Import Tkinter modules for creating graphical user interface dialogs.

# Define the regular expression pattern for the required file naming convention.
# The naming convention is: Name_Institute_Date_DataQualifier
# For example: 20241018_JFi_IPAT_DataSet1
pattern = re.compile(r'^\d{8}_[A-Za-z]+_[A-Za-z]+_[A-Za-z0-9]+$')
# This pattern matches names like "YYYYMMDD_Name_Institute_DataQualifier"

class FileNamingHandler(FileSystemEventHandler):
    # Create a subclass of 'FileSystemEventHandler' to define custom behavior when file system events occur.

    def on_created(self, event):
        """
        This method is automatically called by the observer when a new file or directory is created in the monitored path.
        The 'event' parameter contains information about the event, such as the file path.
        """
        if not event.is_directory:
            # Directly call the file check to avoid threading issues in environments like Jupyter Notebook.
            self.check_file_name(event.src_path)

    def check_file_name(self, file_path):
        """
        Checks if the file name matches the required naming convention.
        If not, it prompts the user to rename the file.
        """
        filename = os.path.basename(file_path)
        # Extract the file name from the full file path.

        if not pattern.match(filename):
            # Use the regular expression 'pattern' to check if the file name matches the required naming convention.
            # If it doesn't match, proceed to prompt the user to rename the file.
            self.prompt_rename(file_path, filename)

    def prompt_rename(self, file_path, filename):
        """
        Prompts the user to rename the file using a graphical dialog.
        If the user cancels the renaming process, move the file to a 'rename' folder.
        """
        root = tk.Tk()
        root.withdraw()  # Hide the root window.
        # Initialize Tkinter and hide the main window since we only need to display dialogs.

        # Prepare a warning message to inform the user about the invalid file name.
        message = (
            f"The file '{filename}' does not adhere to the naming convention.\n"
            f"The required naming format is: YYYYMMDD_Name_Institute_DataQualifier"
        )

        # Display a warning message box to the user.
        messagebox.showwarning("Invalid File Name", message)

        # Get the file extension.
        base_name, extension = os.path.splitext(filename)

        # Loop until the user provides a valid filename or cancels.
        new_name = base_name
        while True:
            # Prompt the user to enter a new file name, without changing the extension.
            new_base_name = simpledialog.askstring(
                "Rename File",
                "Enter a new file name (format: YYYYMMDD_Name_Institute_DataQualifier):",
                initialvalue=new_name
            )

            if not new_base_name:
                # If the user cancels the dialog, move the file to a 'rename' folder.
                dir_path = os.path.dirname(file_path)
                rename_folder = os.path.join(dir_path, 'rename')
                # Create the 'rename' folder if it doesn't exist.
                if not os.path.exists(rename_folder):
                    os.makedirs(rename_folder)

                new_path = os.path.join(rename_folder, filename)
                # Handle the case where the file already exists in the 'rename' folder.
                counter = 1
                while os.path.exists(new_path):
                    new_path = os.path.join(rename_folder, f"{base_name}_{counter}{extension}")
                    counter += 1

                # Move the file to the 'rename' folder with a unique name.
                os.rename(file_path, new_path)

                # Inform the user that the file has been moved.
                messagebox.showinfo("File Moved", f"File renaming was cancelled. The file has been moved to: {new_path}")
                root.destroy()
                return

            new_name = f"{new_base_name}{extension}"

            if pattern.match(new_base_name):
                # If the new name matches the required pattern, proceed.
                break
            else:
                # Inform the user that the entered name is still invalid.
                messagebox.showwarning("Invalid File Name", "The new file name does not match the required format. Please try again.")

        dir_path = os.path.dirname(file_path)
        # Get the directory path of the file.

        new_path = os.path.join(dir_path, new_name)
        # Construct the full file path with the new file name.

        try:
            os.rename(file_path, new_path)
            # Attempt to rename the file to the new name.

            # Inform the user that the file has been successfully renamed.
            messagebox.showinfo("Success", f"File renamed to '{new_name}'")
        except Exception as e:
            # If an error occurs during renaming, catch the exception.
            messagebox.showerror("Error", f"Failed to rename file: {e}")
            # Display an error message to the user with the exception details.

        root.destroy()
        # Destroy the Tkinter root window to clean up resources.

if __name__ == "__main__":
    # This block will only execute if the script is run directly.

    path_to_watch = r"D:/Monitored_Folders/SEM"
    # Specify the path of the directory to be monitored.

    event_handler = FileNamingHandler()
    # Create an instance of the custom event handler.

    observer = Observer()
    # Create an Observer object to monitor the file system.

    observer.schedule(event_handler, path=path_to_watch, recursive=False)
    # Schedule the observer to monitor the specified path with the event handler.
    # Set 'recursive=True' if you want to monitor subdirectories as well.

    observer.start()
    # Start the observer thread, which will now monitor the directory for events.

    print(f"Monitoring directory: {path_to_watch}")
    # Inform the user that monitoring has started.

    try:
        while True:
            time.sleep(1)
            # Keep the script running indefinitely.
    except KeyboardInterrupt:
        # If the user presses Ctrl+C, stop the observer.
        observer.stop()
        print("Monitoring stopped.")

    observer.join()
    # Wait for the observer thread to finish before exiting.


Monitoring directory: D:/Monitored_Folders/SEM
Monitoring stopped.


In [4]:
import os
import time
import shutil
import threading
import re
from tkinter import simpledialog, messagebox
import tkinter as tk

# Import necessary modules to interact with the Watchdog script
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# Define a path to be used for testing
TEST_DIRECTORY = "D:/Monitored_Folders/SEM"
RENAME_DIRECTORY = os.path.join(TEST_DIRECTORY, "rename")

# Define the test files that will be used
TEST_FILES = [
    "20241018_JohnDoe_IPAT_DataSet1.txt",  # Valid file name
    "InvalidFileName.txt",                # Invalid file name
    "20241018_Sam_IPAT_.txt",             # Invalid, missing data qualifier
    "20241018_Sam_IPAT_DataSet2.txt"      # Valid file name
]

# Define the naming convention pattern
pattern = re.compile(r'^\d{8}_[A-Za-z]+_[A-Za-z]+_[A-Za-z0-9]+$')

# Create a custom event handler for testing that extends the original functionality
class FileNamingHandler(FileSystemEventHandler):
    # Create a subclass of 'FileSystemEventHandler' to define custom behavior when file system events occur.

    def on_created(self, event):
        """
        This method is automatically called by the observer when a new file or directory is created in the monitored path.
        The 'event' parameter contains information about the event, such as the file path.
        """
        if not event.is_directory:
            # Run the file check in the main thread using `after` to prevent Tkinter issues with threading.
            threading.Thread(target=self.check_file_name, args=(event.src_path,)).start()

    def check_file_name(self, file_path):
        """
        Checks if the file name matches the required naming convention.
        If not, it prompts the user to rename the file.
        """
        filename = os.path.basename(file_path)
        # Extract the file name from the full file path.

        if not pattern.match(filename):
            # Use the regular expression 'pattern' to check if the file name matches the required naming convention.
            # If it doesn't match, proceed to prompt the user to rename the file.
            self.prompt_rename(file_path, filename)

    def prompt_rename(self, file_path, filename):
        """
        Prompts the user to rename the file using a graphical dialog.
        If the user cancels the renaming process, move the file to a 'rename' folder.
        """
        root = tk.Tk()
        root.withdraw()  # Hide the root window.
        # Initialize Tkinter and hide the main window since we only need to display dialogs.

        def run_dialog():
            # Prepare a warning message to inform the user about the invalid file name.
            message = (
                f"The file '{filename}' does not adhere to the naming convention.\n"
                f"The required naming format is: YYYYMMDD_Name_Institute_DataQualifier"
            )

            # Display a warning message box to the user.
            messagebox.showwarning("Invalid File Name", message)

            # Get the file extension.
            base_name, extension = os.path.splitext(filename)

            # Loop until the user provides a valid filename or cancels.
            new_name = None
            while True:
                # Prompt the user to enter a new file name, without changing the extension.
                new_base_name = simpledialog.askstring(
                    "Rename File",
                    "Enter a new file name (format: YYYYMMDD_Name_Institute_DataQualifier):",
                    initialvalue=base_name
                )

                if not new_base_name:
                    # If the user cancels the dialog, move the file to a 'rename' folder.
                    dir_path = os.path.dirname(file_path)
                    rename_folder = os.path.join(dir_path, 'rename')
                    # Create the 'rename' folder if it doesn't exist.
                    if not os.path.exists(rename_folder):
                        os.makedirs(rename_folder)

                    new_path = os.path.join(rename_folder, filename)
                    # Handle the case where the file already exists in the 'rename' folder.
                    counter = 1
                    while os.path.exists(new_path):
                        new_path = os.path.join(rename_folder, f"{base_name}_{counter}{extension}")
                        counter += 1

                    # Move the file to the 'rename' folder with a unique name.
                    os.rename(file_path, new_path)

                    # Inform the user that the file has been moved.
                    messagebox.showinfo("File Moved", f"File renaming was cancelled. The file has been moved to: {new_path}")
                    root.destroy()
                    return

                new_name = f"{new_base_name}{extension}"

                if pattern.match(new_base_name):
                    # If the new name matches the required pattern, proceed.
                    break
                else:
                    # Inform the user that the entered name is still invalid.
                    messagebox.showwarning("Invalid File Name", "The new file name does not match the required format. Please try again.")

            dir_path = os.path.dirname(file_path)
            # Get the directory path of the file.

            new_path = os.path.join(dir_path, new_name)
            # Construct the full file path with the new file name.

            try:
                os.rename(file_path, new_path)
                # Attempt to rename the file to the new name.

                # Inform the user that the file has been successfully renamed.
                messagebox.showinfo("Success", f"File renamed to '{new_name}'")
            except Exception as e:
                # If an error occurs during renaming, catch the exception.
                messagebox.showerror("Error", f"Failed to rename file: {e}")
                # Display an error message to the user with the exception details.

            root.destroy()
            # Destroy the Tkinter root window to clean up resources.

        root.after(0, run_dialog)
        root.mainloop()

# Function to clean up the test directory before and after running tests
def clean_test_directory():
    # Remove all files and the rename folder in the test directory
    if os.path.exists(TEST_DIRECTORY):
        for root, dirs, files in os.walk(TEST_DIRECTORY):
            for file in files:
                os.remove(os.path.join(root, file))
            for dir in dirs:
                shutil.rmtree(os.path.join(root, dir))

# Function to simulate file creation in the test directory
def simulate_file_creation():
    time.sleep(1)  # Pause to let observer start
    # Create test files in the test directory
    for test_file in TEST_FILES:
        test_path = os.path.join(TEST_DIRECTORY, test_file)
        with open(test_path, 'w') as f:
            f.write("Test content")
        time.sleep(1)  # Wait before creating the next file

# Function to run the Watchdog observer for testing
def run_watchdog_test():
    clean_test_directory()  # Clean up the directory before starting

    # Create test directory if it doesn't exist
    if not os.path.exists(TEST_DIRECTORY):
        os.makedirs(TEST_DIRECTORY)

    # Create an instance of the custom event handler for testing
    event_handler = FileNamingHandler()
    observer = Observer()
    observer.schedule(event_handler, path=TEST_DIRECTORY, recursive=False)
    observer.start()

    print(f"Monitoring directory for testing: {TEST_DIRECTORY}")

    try:
        # Start a separate thread to simulate file creation during testing
        threading.Thread(target=simulate_file_creation).start()

        # Keep the observer running
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        observer.stop()
        print("Testing stopped.")
    finally:
        observer.join()
        clean_test_directory()  # Clean up after testing is done

if __name__ == "__main__":
    run_watchdog_test()


Monitoring directory for testing: D:/Monitored_Folders/SEM
Testing stopped.
